## 개별 노드(에이전트) 테스트

<div style="text-align: right"> <b>KMYU</b></div>
<div style="text-align: right"> Initial issue : 2026.03.20 </div>
<div style="text-align: right"> last update : 2026.03.20 </div>

개정 이력  
- `2026.03.20` : 노트북 최초 작성

**사전 요구사항:**
- PostgreSQL 실행 (`make docker-up && make migrate && make seed`)
- `.env` 파일에 API 키 설정 (OpenAI/Anthropic, Pinecone)
- 각 노드는 순서대로 실행 (이전 노드의 출력이 다음 노드의 입력)

In [ ]:
import sys

sys.path.insert(0, "../../")

from dotenv import load_dotenv

load_dotenv()

### 1. 환경 설정 및 벡터 스토어 초기화

In [ ]:
import json
import uuid

from backend.app.agent.graph import hold_verdict_node
from backend.app.agent.nodes import (
    make_deal_structuring_node,
    make_final_verdict_node,
    make_resource_estimation_node,
    make_risk_analysis_node,
    make_scoring_node,
    make_similar_project_node,
)
from backend.app.db.vector_store import CompanyContextStore, ProjectHistoryStore

# 벡터 스토어 인스턴스 생성
context_store = CompanyContextStore()
project_store = ProjectHistoryStore()

print("벡터 스토어 초기화 완료")

### 2. 샘플 딜 입력 준비

In [ ]:
# 테스트용 deal_id 생성
test_deal_id = str(uuid.uuid4())
print(f"Test Deal ID: {test_deal_id}")

# 정상 딜 입력 (충분한 정보 포함)
sample_deal_input = """고객사: 메가테크 (대기업, 제조업)
프로젝트: 공장 설비 예지보전 AI 시스템 개발
배경: 기존 정기 점검 체계에서 AI 기반 예측 유지보수로 전환하여 설비 가동률 향상 및 유지보수 비용 절감
기술 요구사항: Python, TensorFlow, 시계열 분석, IoT 센서 데이터 처리, 대시보드 (React)
예상 규모: 2억원
기간: 5개월
결제 조건: 착수금 30%, 중간 40%, 완료 30%
특이사항: 공장 내 보안 네트워크 환경에서만 운영, 온프레미스 배포 필수"""

# 초기 state 구성
state = {
    "deal_id": test_deal_id,
    "deal_input": sample_deal_input,
}
print(f"초기 state keys: {list(state.keys())}")

### 3. deal_structuring 노드

In [ ]:
deal_structuring = make_deal_structuring_node(context_store)
result = await deal_structuring(state)

print("=== deal_structuring 결과 ===")
print(f"status: {result.get('status')}")
print("structured_deal:")
print(json.dumps(result.get("structured_deal", {}), ensure_ascii=False, indent=2))

In [ ]:
# state 누적
state.update(result)

# missing_fields 확인 (Hold 분기 판단 기준)
missing = state.get("structured_deal", {}).get("missing_fields", [])
print(f"Missing fields: {missing} (count: {len(missing)})")
print(f"Hold 여부: {'Hold (≥3 missing)' if len(missing) >= 3 else 'Continue'}")

### 4. hold_verdict 분기 테스트

In [ ]:
# missing_fields가 3개 이상인 경우를 시뮬레이션
hold_state = {
    "deal_id": test_deal_id,
    "deal_input": "AI 프로젝트를 하고 싶습니다.",  # 정보 부족
    "structured_deal": {
        "customer_name": None,
        "customer_size": None,
        "customer_industry": None,
        "project_summary": "AI 프로젝트",
        "tech_requirements": [],
        "expected_amount": None,
        "duration_months": None,
        "payment_terms": None,
        "special_notes": None,
        "missing_fields": ["customer_name", "customer_size", "customer_industry", "expected_amount", "duration_months"],
    },
}

hold_result = hold_verdict_node(hold_state)

print("=== hold_verdict 결과 ===")
print(f"verdict: {hold_result['verdict']}")
print(f"total_score: {hold_result['total_score']}")
print(f"final_report:\n{hold_result['final_report']}")

### 5. scoring 노드

In [ ]:
scoring = make_scoring_node(context_store)
result = await scoring(state)

print("=== scoring 결과 ===")
print(f"total_score: {result.get('total_score')}")
print(f"verdict: {result.get('verdict')}")
print("\nscores:")
for s in result.get("scores", []):
    print(f"  - {s['criterion']}: {s['score']}점 (가중치 {s['weight']}, 가중점수 {s['weighted_score']}) — {s['rationale']}")

In [ ]:
# state 누적
state.update(result)
print(f"현재 state keys: {list(state.keys())}")

### 6. resource_estimation 노드

In [ ]:
resource_estimation = make_resource_estimation_node(project_store)
result = await resource_estimation(state)

print("=== resource_estimation 결과 ===")
print(json.dumps(result.get("resource_estimate", {}), ensure_ascii=False, indent=2))

In [ ]:
state.update(result)
print(f"현재 state keys: {list(state.keys())}")

### 7. risk_analysis 노드

In [ ]:
risk_analysis = make_risk_analysis_node(context_store)
result = await risk_analysis(state)

print("=== risk_analysis 결과 ===")
for r in result.get("risks", []):
    print(f"  - [{r.get('level', 'N/A')}] {r.get('category', 'N/A')}: {r.get('item', '')} — {r.get('description', '')}")

In [ ]:
state.update(result)
print(f"현재 state keys: {list(state.keys())}")

### 8. similar_project 노드

In [ ]:
similar_project = make_similar_project_node(project_store)
result = await similar_project(state)

print("=== similar_project 결과 ===")
similar_projects = result.get("similar_projects", [])
if similar_projects:
    print(json.dumps(similar_projects, ensure_ascii=False, indent=2))
else:
    print("유사 프로젝트 없음 (벡터 스토어에 데이터가 없는 경우 정상)")

In [ ]:
state.update(result)
print(f"현재 state keys: {list(state.keys())}")

### 9. final_verdict 노드

In [ ]:
final_verdict = make_final_verdict_node()
result = await final_verdict(state)

print("=== final_verdict 결과 ===")
print(f"status: {result.get('status')}")
print("\nfinal_report (처음 500자):")
print(result.get("final_report", "")[:500], "...")

In [ ]:
# 최종 리포트 마크다운 렌더링
from IPython.display import Markdown, display

display(Markdown(result.get("final_report", "")))

### 10. 전체 파이프라인 state 요약

In [ ]:
state.update(result)

print("=== 최종 State 요약 ===")
print(f"deal_id: {state['deal_id']}")
print(f"status: {state.get('status')}")
print(f"verdict: {state.get('verdict')}")
print(f"total_score: {state.get('total_score')}")
print(f"scores count: {len(state.get('scores', []))}")
print(f"risks count: {len(state.get('risks', []))}")
print(f"similar_projects count: {len(state.get('similar_projects', []))}")
print(f"resource_estimate keys: {list(state.get('resource_estimate', {}).keys())}")
print(f"final_report length: {len(state.get('final_report', ''))} chars")
print(f"errors: {state.get('errors', [])}")